In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from xgboost import XGBRegressor

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("Conectando con Google Drive...")
drive.mount('/content/drive')

# Definir y crear la ruta de la carpeta si no existe
ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_csv = os.path.join(ruta_drive, 'datos_solares_avanzados.csv')

# =====================================================================
# 1. GENERACIÓN DE LA BASE DE DATOS AMPLIADA
# =====================================================================
print("\n1. Creando base de datos con variables climáticas...")
np.random.seed(42)
fechas = pd.date_range(start="2026-06-01", end="2026-06-30", freq="h")
horas = fechas.hour

# Simulación de variables meteorológicas
radiacion = np.where((horas > 5) & (horas < 20), np.sin((horas - 5) / 14 * np.pi) * 900 + np.random.normal(0, 40, len(fechas)), 0)
radiacion = np.clip(radiacion, 0, None)

temperatura = 22 + np.sin((horas - 8) / 24 * 2 * np.pi) * 12 + np.random.normal(0, 1.5, len(fechas))
viento = np.abs(np.random.normal(4, 2, len(fechas)))
nubosidad = np.clip(np.random.beta(2, 5, len(fechas)) * 100 + np.where(radiacion == 0, 20, 0), 0, 100)
humedad = np.clip(100 - (temperatura * 2) - (radiacion * 0.03) + np.random.normal(0, 5, len(fechas)), 10, 100)

# Variable Objetivo: Generación MW
eficiencia = 0.16 - (temperatura - 25) * 0.003
generacion = (radiacion * (1 - nubosidad/100) * eficiencia * 0.06) + np.random.normal(0, 0.3, len(fechas))
generacion = np.where(radiacion == 0, 0, np.clip(generacion, 0, None))

# Guardar DataFrame directamente en la ruta de tu Drive
df_raw = pd.DataFrame({
    'Fecha_Hora': fechas,
    'Radiacion_Solar': radiacion,
    'Temperatura_Ambiente': temperatura,
    'Velocidad_Viento': viento,
    'Nubosidad_Porcentaje': nubosidad,
    'Humedad_Relativa': humedad,
    'Generacion_MW': generacion
})
df_raw.to_csv(ruta_csv, index=False)
print(f"¡Archivo guardado exitosamente en: {ruta_csv}!\n")

# =====================================================================
# 2. INGENIERÍA DE VARIABLES AVANZADA (FEATURE ENGINEERING)
# =====================================================================
print("2. Leyendo desde Drive y ejecutando Ingeniería de Variables...")
df = pd.read_csv(ruta_csv)
df['Fecha_Hora'] = pd.to_datetime(df['Fecha_Hora'])
df.set_index('Fecha_Hora', inplace=True)

# A. Transformación Cíclica del Tiempo
df['Hora_Sin'] = np.sin(2 * np.pi * df.index.hour / 24.0)
df['Hora_Cos'] = np.cos(2 * np.pi * df.index.hour / 24.0)
df['Mes_Sin'] = np.sin(2 * np.pi * df.index.month / 12.0)
df['Mes_Cos'] = np.cos(2 * np.pi * df.index.month / 12.0)

# B. Variables Rezagadas (Lags)
df['Radiacion_Lag1'] = df['Radiacion_Solar'].shift(1)
df['Radiacion_Lag2'] = df['Radiacion_Solar'].shift(2)
df['Generacion_Lag1'] = df['Generacion_MW'].shift(1)

# C. Diferencias Temporales
df['Radiacion_Diff1'] = df['Radiacion_Solar'].diff(1)

# D. Medias Móviles (Rolling Windows)
df['Temp_Media_3h'] = df['Temperatura_Ambiente'].rolling(window=3).mean()
df['Nubes_Media_3h'] = df['Nubosidad_Porcentaje'].rolling(window=3).mean()

df.dropna(inplace=True)

# =====================================================================
# 3. ENTRENAMIENTO DEL MODELO (XGBOOST)
# =====================================================================
print("3. Entrenando el modelo XGBoost...")
X = df.drop(columns=['Generacion_MW'])
y = df['Generacion_MW']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

modelo_avanzado = XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
modelo_avanzado.fit(X_train, y_train)
predicciones = modelo_avanzado.predict(X_test)

# =====================================================================
# 4. EVALUACIÓN Y GRÁFICAS DE DIAGNÓSTICO
# =====================================================================
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
r2 = r2_score(y_test, predicciones)

print(f"\n================ METRICAS DEL MODELO ================")
print(f"Error Absoluto Medio (MAE):         {mae:.2f} MW")
print(f"Raíz del Error Cuadrático (RMSE):   {rmse:.2f} MW")
print(f"Coeficiente de Determinación (R²):  {r2:.4f}")
print(f"=====================================================\n")

# --- GRÁFICA 1: Serie Temporal (Real vs Predicho) ---
plt.figure(figsize=(15, 5))
plt.plot(y_test.index[:96], y_test.values[:96], label='Generación Real (MW)', color='#1f77b4', linewidth=2, marker='o')
plt.plot(y_test.index[:96], predicciones[:96], label='Predicción XGBoost (MW)', color='#ff7f0e', linestyle='--', linewidth=2, marker='x')
plt.title('Comparativa de Generación Fotovoltaica: Real vs Predicción (Muestra de 4 días)', fontsize=14)
plt.ylabel('Potencia (MW)')
plt.xlabel('Fecha y Hora')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()
print(" ")
print(" ")
# --- GRÁFICA 2: Importancia de las Variables ---
importancias = modelo_avanzado.feature_importances_
indices = np.argsort(importancias)[::-1]

plt.figure(figsize=(12, 5))
sns.barplot(x=importancias[indices], y=X.columns[indices], palette='viridis', hue=X.columns[indices], legend=False)
plt.title('Importancia de las Variables en el Modelo Predictivo', fontsize=14)
plt.xlabel('Importancia Relativa')
plt.ylabel('Variables (Features)')
plt.grid(True, axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()
print(" ")
print(" ")
# --- GRÁFICA 3: Análisis de Residuos ---
residuos = y_test - predicciones

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(predicciones, residuos, alpha=0.4, color='purple')
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Residuos vs Valores Predichos')
plt.xlabel('Predicciones (MW)')
plt.ylabel('Error (Real - Predicho)')

plt.subplot(1, 2, 2)
sns.histplot(residuos, kde=True, color='teal')
plt.axvline(x=0, color='r', linestyle='--')
plt.title('Distribución de los Errores (Residuos)')
plt.xlabel('Magnitud del Error (MW)')
plt.tight_layout()
plt.show()


In [4]:
df.head()

,Radiacion_Solar,Temperatura_Ambiente,Velocidad_Viento,Nubosidad_Porcentaje,Humedad_Relativa,Generacion_MW,Hora_Sin,Hora_Cos,Mes_Sin,Mes_Cos,Radiacion_Lag1,Radiacion_Lag2,Generacion_Lag1,Radiacion_Diff1,Temp_Media_3h,Nubes_Media_3h
Fecha_Hora,,,,,,,,,,,,,,,,
2026-06-01 02:00:00,0.000000,9.722647,1.305747,57.060415,87.281613,0.000000,0.500000,8.660254e-01,1.224647e-16,-1.0,0.0,0.0,0.0,0.000000,10.416201,59.391191
2026-06-01 03:00:00,0.000000,9.624806,2.056772,44.928538,80.208365,0.000000,0.707107,7.071068e-01,1.224647e-16,-1.0,0.0,0.0,0.0,0.000000,10.380794,56.108021
2026-06-01 04:00:00,0.000000,13.181209,6.400828,42.281934,80.511503,0.000000,0.866025,5.000000e-01,1.224647e-16,-1.0,0.0,0.0,0.0,0.000000,10.842887,48.090295
2026-06-01 05:00:00,0.000000,12.458203,2.686211,38.995471,73.508092,0.000000,0.965926,2.588190e-01,1.224647e-16,-1.0,0.0,0.0,0.0,0.000000,11.754739,42.068647
2026-06-01 06:00:00,263.437353,13.887308,1.906178,46.661010,68.954673,2.037214,1.000000,6.123234e-17,1.224647e-16,-1.0,0.0,0.0,0.0,263.437353,13.175573,42.646138


In [5]:
df.describe()

,Radiacion_Solar,Temperatura_Ambiente,Velocidad_Viento,Nubosidad_Porcentaje,Humedad_Relativa,Generacion_MW,Hora_Sin,Hora_Cos,Mes_Sin,Mes_Cos,Radiacion_Lag1,Radiacion_Lag2,Generacion_Lag1,Radiacion_Diff1,Temp_Media_3h,Nubes_Media_3h
count,695.000000,695.000000,695.000000,695.000000,695.000000,695.000000,695.000000,6.950000e+02,6.950000e+02,695.0,695.000000,695.000000,695.000000,6.950000e+02,695.000000,695.000000
mean,335.086346,22.134534,4.128573,37.451336,46.349628,2.098182,-0.000372,-1.389821e-03,1.224647e-16,-1.0,335.086346,335.086346,2.098182,-3.271564e-16,22.131254,37.422845
std,354.728223,8.625253,1.893349,18.678428,26.127736,2.277447,0.708057,7.071737e-01,1.480179e-31,0.0,354.728223,354.728223,2.277447,1.154146e+02,8.350440,13.162765
min,0.000000,6.578008,0.007725,0.908321,10.000000,0.000000,-1.000000,-1.000000e+00,1.224647e-16,-1.0,0.000000,0.000000,0.000000,-3.215002e+02,8.779353,4.349883
25%,0.000000,13.787226,2.782395,24.454497,20.034207,0.000000,-0.707107,-7.071068e-01,1.224647e-16,-1.0,0.000000,0.000000,0.000000,-4.611033e+01,13.847341,27.222105
50%,209.232540,21.771295,4.037699,36.309610,49.300449,1.237448,0.000000,-1.836970e-16,1.224647e-16,-1.0,209.232540,209.232540,1.237448,0.000000e+00,22.090043,36.882183
75%,712.618505,30.445965,5.417756,48.417337,70.767663,4.138879,0.707107,7.071068e-01,1.224647e-16,-1.0,712.618505,712.618505,4.138879,4.548935e+01,30.382824,45.520784
max,986.241888,37.381509,10.386215,100.000000,98.233805,7.538352,1.000000,1.000000e+00,1.224647e-16,-1.0,986.241888,986.241888,7.538352,3.191071e+02,35.650136,74.568639


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 695 entries, 2026-06-01 02:00:00 to 2026-06-30 00:00:00
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Radiacion_Solar       695 non-null    float64
 1   Temperatura_Ambiente  695 non-null    float64
 2   Velocidad_Viento      695 non-null    float64
 3   Nubosidad_Porcentaje  695 non-null    float64
 4   Humedad_Relativa      695 non-null    float64
 5   Generacion_MW         695 non-null    float64
 6   Hora_Sin              695 non-null    float64
 7   Hora_Cos              695 non-null    float64
 8   Mes_Sin               695 non-null    float64
 9   Mes_Cos               695 non-null    float64
 10  Radiacion_Lag1        695 non-null    float64
 11  Radiacion_Lag2        695 non-null    float64
 12  Generacion_Lag1       695 non-null    float64
 13  Radiacion_Diff1       695 non-null    float64
 14  Temp_Media_3h         695 non-null   

In [14]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from xgboost import XGBRegressor

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("Conectando con Google Drive...")
drive.mount('/content/drive')

# Definir la ruta compartida de tus proyectos
ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_csv = os.path.join(ruta_drive, 'datos_demanda_avanzados.csv')

# =====================================================================
# 1. GENERACIÓN DE LA BASE DE DATOS DE DEMANDA (datos_demanda_avanzados.csv)
# =====================================================================
print("\n1. Creando base de datos de demanda eléctrica...")
np.random.seed(24)
fechas = pd.date_range(start="2026-01-01", end="2026-03-31", freq="h")
horas = fechas.hour
dias_semana = fechas.dayofweek

# Simulación de temperatura (Invierno)
temperatura = 8 + np.sin((horas - 8) / 24 * 2 * np.pi) * 6 + np.random.normal(0, 2, len(fechas))

# Variables de Calendario
es_fin_semana = np.where(dias_semana >= 5, 1, 0)
festivos_fechas = ['2026-01-01', '2026-01-06', '2026-03-19']
es_festivo = np.where(fechas.strftime('%Y-%m-%d').isin(festivos_fechas), 1, 0)

# Construcción de la Variable Objetivo: Demanda MW
demanda_base = 200 + np.sin((horas - 6) / 24 * 2 * np.pi) * 40 + np.sin((horas - 15) / 24 * 4 * np.pi) * 20
factor_calendario = np.ones(len(fechas))
factor_calendario = np.where(es_fin_semana == 1, 0.75, factor_calendario)
factor_calendario = np.where(es_festivo == 1, 0.70, factor_calendario)
efecto_termico = np.where(temperatura < 18, (18 - temperatura) ** 1.5 * 4, 0)

demanda = (demanda_base * factor_calendario) + efecto_termico + np.random.normal(0, 5, len(fechas))
demanda = np.clip(demanda, 50, None)

# Guardar en DataFrame y exportar a Drive
df_raw = pd.DataFrame({
    'Fecha_Hora': fechas,
    'Temperatura_Ambiente': temperatura,
    'Dia_Semana': dias_semana,
    'Es_Fin_Semana': es_fin_semana,
    'Es_Festivo': es_festivo,
    'Demanda_MW': demanda
})
df_raw.to_csv(ruta_csv, index=False)
print(f"¡Archivo de demanda guardado con éxito en: {ruta_csv}!\n")

# =====================================================================
# 2. INGENIERÍA DE VARIABLES (FEATURE ENGINEERING)
# =====================================================================
print("2. Leyendo desde Drive y ejecutando Ingeniería de Variables...")
df = pd.read_csv(ruta_csv)
df['Fecha_Hora'] = pd.to_datetime(df['Fecha_Hora'])
df.set_index('Fecha_Hora', inplace=True)

# Transformación Cíclica del Tiempo
df['Hora_Sin'] = np.sin(2 * np.pi * df.index.hour / 24.0)
df['Hora_Cos'] = np.cos(2 * np.pi * df.index.hour / 24.0)
df['Dia_Semana_Sin'] = np.sin(2 * np.pi * df['Dia_Semana'] / 7.0)
df['Dia_Semana_Cos'] = np.cos(2 * np.pi * df['Dia_Semana'] / 7.0)

# Variables Rezagadas (Lags) y Medias Móviles
df['Demanda_Lag1'] = df['Demanda_MW'].shift(1)
df['Demanda_Lag2'] = df['Demanda_MW'].shift(2)
df['Demanda_Lag24'] = df['Demanda_MW'].shift(24)
df['Temp_Lag1'] = df['Temperatura_Ambiente'].shift(1)
df['Temp_Media_4h'] = df['Temperatura_Ambiente'].rolling(window=4).mean()

df.dropna(inplace=True)

# =====================================================================
# 3. ENTRENAMIENTO DEL MODELO (XGBOOST)
# =====================================================================
print("3. Entrenando el modelo XGBoost...")
X = df.drop(columns=['Demanda_MW'])
y = df['Demanda_MW']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

modelo_demanda = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.9,
    random_state=42
)
modelo_demanda.fit(X_train, y_train)
predicciones = modelo_demanda.predict(X_test)

# =====================================================================
# 4. EVALUACIÓN Y MÉTRICAS
# =====================================================================
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
r2 = r2_score(y_test, predicciones)

print(f"\n================ METRICAS DEL MODELO ================")
print(f"Error Absoluto Medio (MAE):         {mae:.2f} MW")
print(f"Raíz del Error Cuadrático (RMSE):   {rmse:.2f} MW")
print(f"Coeficiente de Determinación (R²):  {r2:.4f}")
print(f"=====================================================\n")

# =====================================================================
# 5. VISUALIZACIÓN INTERACTIVA CON PLOTLY MODIFICADA
# =====================================================================
print("4. Generando paneles dinámicos interactivos actualizados...")

# --- GRÁFICA 1: Serie Temporal Interactiva (Real vs Predicho) ---
indices_muestra = y_test.index[:168]
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=indices_muestra, y=y_test.values[:168], mode='lines+markers', name='Demanda Real', line=dict(color='#2ca02c', width=2)))
fig1.add_trace(go.Scatter(x=indices_muestra, y=predicciones[:168], mode='lines+markers', name='Predicción XGBoost', line=dict(color='#d62728', width=2, dash='dash')))
fig1.update_layout(
    title='Predicción de Demanda Eléctrica: Real vs Predicción (Muestra de 1 Semana)',
    xaxis_title='Fecha y Hora',
    yaxis_title='Carga de la Red (MW)',
    template='plotly_white',
    hovermode='x unified'
)
fig1.show()

# --- GRÁFICA 2: Importancia de las Variables (Valores fijos al final de la barra + formato corregido) ---
importancias = modelo_demanda.feature_importances_
indices_imp = np.argsort(importancias)

# Convertimos la importancia relativa a porcentaje puro
valores_porcentaje = importancias[indices_imp] * 100

df_imp = pd.DataFrame({
    'Variable': X.columns[indices_imp],
    'Importancia': valores_porcentaje
})

# Generamos las etiquetas estáticas en Python formateadas exactamente a 4 decimales + %
etiquetas_fijas = [f"{v:.4f}%" for v in valores_porcentaje]

fig2 = px.bar(
    df_imp,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='Importancia de las Variables en el Modelo de Demanda',
    labels={'Importancia': 'Importancia Relativa (%)', 'Variable': 'Variables (Features)'},
    color='Importancia',
    color_continuous_scale='Magma',
    text=etiquetas_fijas # Pasamos el arreglo de strings mapeado manualmente
)

# Ajustamos la posición del texto afuera de las barras y el formato del Hover
fig2.update_traces(
    textposition='outside',
    hovertemplate='Variable: %{y}<br>Importancia: %{x:.4f}%<extra></extra>'
)

fig2.update_layout(
    template='plotly_white',
    coloraxis_showscale=False,
    xaxis=dict(ticksuffix="%", range=[0, max(valores_porcentaje) * 1.15]) # Añadimos margen extra a la derecha para que no se corten las etiquetas de texto
)
fig2.show()

# --- GRÁFICA 3: Análisis de Residuos ---
residuos = y_test - predicciones

df_residuos = pd.DataFrame({
    'Prediccion_MW': predicciones,
    'Error_MW': residuos
})

fig3 = px.scatter(
    df_residuos,
    x='Prediccion_MW',
    y='Error_MW',
    title='Residuos vs Valores Predichos (Demanda)',
    labels={'Prediccion_MW': 'Predicciones del Modelo (MW)', 'Error_MW': 'Error de Predicción (Real - Predicho)'},
    opacity=0.4
)

fig3.update_traces(
    mode='markers',
    marker=dict(color='blue'),
    hovertemplate='Predicción: %{x:.4f} MW<br>Error: %{y:.4f} MW<extra></extra>'
)

fig3.add_shape(type="line", x0=min(predicciones), y0=0, x1=max(predicciones), y1=0, line=dict(color="red", width=2, dash="dash"))
fig3.update_layout(template='plotly_white')
fig3.show()

# --- GRÁFICA 4: Histograma Distribución de Errores ---
fig4 = px.histogram(residuos, nbins=40, title='Distribución de los Errores de Demanda', labels={'value': 'Magnitud del Error (MW)'}, color_discrete_sequence=['crimson'])
fig4.add_shape(type="line", x0=0, y0=0, x1=0, y1=1, xref="x", yref="paper", line=dict(color="black", width=2))
fig4.update_layout(template='plotly_white', showlegend=False, yaxis_title='Frecuencia')
fig4.show()


Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1. Creando base de datos de demanda eléctrica...
¡Archivo de demanda guardado con éxito en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/datos_demanda_avanzados.csv!

2. Leyendo desde Drive y ejecutando Ingeniería de Variables...
3. Entrenando el modelo XGBoost...

================ METRICAS DEL MODELO ================
Error Absoluto Medio (MAE):         4.98 MW
Raíz del Error Cuadrático (RMSE):   6.47 MW
Coeficiente de Determinación (R²):  0.9922

4. Generando paneles dinámicos interactivos actualizados...


In [16]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from xgboost import XGBRegressor

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("Conectando con Google Drive...")
drive.mount('/content/drive')

# Definir la ruta compartida de tus proyectos
ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_csv = os.path.join(ruta_drive, 'datos_precios_mercado.csv')

# =====================================================================
# 1. GENERACIÓN DE LA BASE DE DATOS DE PRECIOS (datos_precios_mercado.csv)
# =====================================================================
print("\n1. Creando base de datos sintética del Mercado Eléctrico Diario...")
np.random.seed(99)
fechas = pd.date_range(start="2026-06-01", end="2026-06-30", freq="h")
horas = fechas.hour
dias_semana = fechas.dayofweek

# Simulación de las variables de nuestros modelos previos (Puntos 1 y 2)
demanda = 220 + np.sin((horas - 7) / 24 * 2 * np.pi) * 50 + np.where(dias_semana >= 5, -40, 0) + np.random.normal(0, 5, len(fechas))
solar = np.where((horas > 5) & (horas < 20), np.sin((horas - 5) / 14 * np.pi) * 120 + np.random.normal(0, 5, len(fechas)), 0)
solar = np.clip(solar, 0, None)

# Cálculo del Margen de Reserva (Demanda neta)
margen_reserva = demanda - solar

# Variables de Commodities (Gas y CO2)
precio_gas = 35 + np.cumsum(np.random.normal(0, 0.4, len(fechas)))
precio_co2 = 70 + np.cumsum(np.random.normal(0, 0.2, len(fechas)))

# Variable Objetivo: Precio del Pool (€/MWh)
precio_base_gas = (precio_gas * 2) + (precio_co2 * 0.4)
componente_escasez = np.where(margen_reserva > 180, ((margen_reserva - 180) ** 1.8) * 1.5, 0)
componente_solar = np.where((solar > 80) & (demanda < 200), -40, 0)

precio_pool = precio_base_gas + componente_escasez + componente_solar + np.random.normal(0, 4, len(fechas))
precio_pool = np.clip(precio_pool, 0, None)

# Guardar en Drive
df_raw = pd.DataFrame({
    'Fecha_Hora': fechas,
    'Demanda_MW': demanda,
    'Generacion_Solar_MW': solar,
    'Margen_Reserva_MW': margen_reserva,
    'Precio_Gas_Euros_MWh': precio_gas,
    'Precio_CO2_Euros_Ton': precio_co2,
    'Precio_Pool_Euros_MWh': precio_pool
})
df_raw.to_csv(ruta_csv, index=False)
print(f"¡Archivo de precios del mercado guardado con éxito en: {ruta_csv}!\n")

# =====================================================================
# 2. INGENIERÍA DE VARIABLES (FEATURE ENGINEERING)
# =====================================================================
print("2. Leyendo desde Drive y ejecutando Ingeniería de Variables Financieras...")
df = pd.read_csv(ruta_csv)
df['Fecha_Hora'] = pd.to_datetime(df['Fecha_Hora'])
df.set_index('Fecha_Hora', inplace=True)

# Transformación Cíclica del Tiempo
df['Hora_Sin'] = np.sin(2 * np.pi * df.index.hour / 24.0)
df['Hora_Cos'] = np.cos(2 * np.pi * df.index.hour / 24.0)

# Variables Rezagadas (Lags)
df['Precio_Lag1'] = df['Precio_Pool_Euros_MWh'].shift(1)
df['Precio_Lag2'] = df['Precio_Pool_Euros_MWh'].shift(2)
df['Precio_Lag24'] = df['Precio_Pool_Euros_MWh'].shift(24)

# Características de Tendencia
df['Variacion_Margen_3h'] = df['Margen_Reserva_MW'].diff(3)

df.dropna(inplace=True)

# =====================================================================
# 3. ENTRENAMIENTO DEL MODELO (XGBOOST)
# =====================================================================
print("3. Entrenando el modelo XGBoost para Precios de la Energía...")
X = df.drop(columns=['Precio_Pool_Euros_MWh'])
y = df['Precio_Pool_Euros_MWh']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

modelo_precios = XGBRegressor(
    n_estimators=250,
    learning_rate=0.04,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
modelo_precios.fit(X_train, y_train)
predicciones = modelo_precios.predict(X_test)

# =====================================================================
# 4. EVALUACIÓN Y MÉTRICAS
# =====================================================================
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
r2 = r2_score(y_test, predicciones)

print(f"\n================ METRICAS DEL MODELO ================")
print(f"Error Absoluto Medio (MAE):         {mae:.2f} €/MWh")
print(f"Raíz del Error Cuadrático (RMSE):   {rmse:.2f} €/MWh")
print(f"Coeficiente de Determinación (R²):  {r2:.4f}")
print(f"=====================================================\n")

# =====================================================================
# 5. VISUALIZACIÓN INTERACTIVA CON PLOTLY MODIFICADA
# =====================================================================
print("4. Generando paneles dinámicos interactivos para Precios...")

# --- GRÁFICA 1: Serie Temporal Interactiva (Real vs Predicho) ---
indices_muestra = y_test.index[:120]
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=indices_muestra, y=y_test.values[:120], mode='lines+markers', name='Precio Real Pool', line=dict(color='#bcbd22', width=2)))
fig1.add_trace(go.Scatter(x=indices_muestra, y=predicciones[:120], mode='lines+markers', name='Predicción XGBoost', line=dict(color='#17becf', width=2, dash='dash')))
fig1.update_layout(
    title='Predicción del Precio de la Electricidad (€/MWh): Real vs Predicción',
    xaxis_title='Fecha y Hora',
    yaxis_title='Precio del Mercado (€/MWh)',
    template='plotly_white',
    hovermode='x unified'
)
fig1.show()
print(" ")
print(" ")
# --- GRÁFICA 2: Importancia de las Variables (Con formato 4 decimales y % afuera) ---
importancias = modelo_precios.feature_importances_
indices_imp = np.argsort(importancias)
valores_porcentaje = importancias[indices_imp] * 100

df_imp = pd.DataFrame({
    'Variable': X.columns[indices_imp],
    'Importancia': valores_porcentaje
})

etiquetas_fijas = [f"{v:.4f}%" for v in valores_porcentaje]

fig2 = px.bar(
    df_imp,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='Importancia de las Variables en la Fijación de Precios',
    labels={'Importancia': 'Importancia Relativa (%)', 'Variable': 'Variables (Features)'},
    color='Importancia',
    color_continuous_scale='Cividis',
    text=etiquetas_fijas
)

fig2.update_traces(
    textposition='outside',
    hovertemplate='Variable: %{y}<br>Importancia: %{x:.4f}%<extra></extra>'
)

fig2.update_layout(
    template='plotly_white',
    coloraxis_showscale=False,
    xaxis=dict(ticksuffix="%", range=[0, max(valores_porcentaje) * 1.15])
)
fig2.show()
print(" ")
print(" ")
# --- GRÁFICA 3: Curva de Oferta (Redondeada a 4 cifras en Hover) ---
fig3 = px.scatter(
    df,
    x='Margen_Reserva_MW',
    y='Precio_Pool_Euros_MWh',
    color='Precio_Gas_Euros_MWh',
    title='Comportamiento Marginalista del Mercado: Precio vs Margen de Reserva',
    labels={
        'Margen_Reserva_MW': 'Margen de Reserva Obligatorio (Demanda - Solar) [MW]',
        'Precio_Pool_Euros_MWh': 'Precio de la Electricidad (€/MWh)',
        'Precio_Gas_Euros_MWh': 'Precio Gas (€/MWh)'
    },
    opacity=0.7,
    color_continuous_scale='Turbo'
)

# Configuración del hover con precisión estricta de 4 decimales
fig3.update_traces(
    hovertemplate='Margen Reserva: %{x:.4f} MW<br>Precio Pool: %{y:.4f} €/MWh<extra></extra>'
)

fig3.update_layout(template='plotly_white')
fig3.show()


Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1. Creando base de datos sintética del Mercado Eléctrico Diario...
¡Archivo de precios del mercado guardado con éxito en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/datos_precios_mercado.csv!

2. Leyendo desde Drive y ejecutando Ingeniería de Variables Financieras...
3. Entrenando el modelo XGBoost para Precios de la Energía...

================ METRICAS DEL MODELO ================
Error Absoluto Medio (MAE):         11.69 €/MWh
Raíz del Error Cuadrático (RMSE):   23.63 €/MWh
Coeficiente de Determinación (R²):  0.9898

4. Generando paneles dinámicos interactivos para Precios...


In [26]:
import os
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from google.colab import drive
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("Conectando con Google Drive...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)

# Generar semillas aleatorias estables
np.random.seed(42)
fechas = pd.date_range(start="2026-06-01", end="2026-06-30", freq="h")
horas = fechas.hour
dias_semana = fechas.dayofweek

# Definir festivos (para el mes de junio no habrá festivos específicos en este ejemplo)
festivos_fechas = [] # No hay festivos en Junio para este ejemplo
es_festivo = np.where(fechas.strftime('%Y-%m-%d').isin(festivos_fechas), 1, 0)

# =====================================================================
# FUNCIÓN AUXILIAR PARA CORREGIR Y RENDERIZAR LA GRÁFICA 2 INTERACTIVA
# =====================================================================
def plot_feature_importance(modelo, columnas, titulo_grafica):
    importancias = modelo.feature_importances_
    indices_imp = np.argsort(importancias)
    valores_porcentaje = importancias[indices_imp] * 100

    df_imp = pd.DataFrame({
        'Variable': [columnas[i] for i in indices_imp],
        'Importancia': valores_porcentaje
    })

    etiquetas_fijas = [f"{v:.4f}%" for v in valores_porcentaje]

    fig = px.bar(
        df_imp, x='Importancia', y='Variable', orientation='h',
        title=titulo_grafica, color='Importancia',
        color_continuous_scale='Viridis', text=etiquetas_fijas
    )
    fig.update_traces(
        textposition='outside',
        hovertemplate='Variable: %{y}<br>Importancia: %{x:.4f}%<extra></extra>'
    )
    fig.update_layout(
        template='plotly_white', coloraxis_showscale=False,
        xaxis=dict(ticksuffix="%", range=[0, max(valores_porcentaje) * 1.15])
    )
    fig.show()

# =====================================================================
# 1. PROCESAMIENTO Y ENTRENAMIENTO: MODELO 1 (SOLAR FOTOVOLTAICA)
# =====================================================================
print("\n--- PROCESANDO MODELO 1: GENERACIÓN SOLAR ---")
radiacion = np.where((horas > 5) & (horas < 20), np.sin((horas - 5) / 14 * np.pi) * 900 + np.random.normal(0, 40, len(fechas)), 0)
radiacion = np.clip(radiacion, 0, None)
temperatura = 22 + np.sin((horas - 8) / 24 * 2 * np.pi) * 12 + np.random.normal(0, 1.5, len(fechas))
viento = np.abs(np.random.normal(4, 2, len(fechas)))
nubosidad = np.clip(np.random.beta(2, 5, len(fechas)) * 100 + np.where(radiacion == 0, 20, 0), 0, 100)
humedad = np.clip(100 - (temperatura * 2) - (radiacion * 0.03) + np.random.normal(0, 5, len(fechas)), 10, 100)

eficiencia = 0.16 - (temperatura - 25) * 0.003
generacion_solar = (radiacion * (1 - nubosidad/100) * eficiencia * 0.06) + np.random.normal(0, 0.3, len(fechas))
generacion_solar = np.where(radiacion == 0, 0, np.clip(generacion_solar, 0, None))

df_solar = pd.DataFrame({
    'Radiacion_Solar': radiacion, 'Temperatura_Ambiente': temperatura, 'Velocidad_Viento': viento,
    'Nubosidad_Porcentaje': nubosidad, 'Humedad_Relativa': humedad, 'Generacion_MW': generacion_solar
}, index=fechas)

df_solar['Hora_Sin'] = np.sin(2 * np.pi * df_solar.index.hour / 24.0)
df_solar['Hora_Cos'] = np.cos(2 * np.pi * df_solar.index.hour / 24.0)
df_solar['Mes_Sin'] = np.sin(2 * np.pi * df_solar.index.month / 12.0)
df_solar['Mes_Cos'] = np.cos(2 * np.pi * df_solar.index.month / 12.0)
df_solar['Radiacion_Lag1'] = df_solar['Radiacion_Solar'].shift(1)
df_solar['Radiacion_Lag2'] = df_solar['Radiacion_Solar'].shift(2)
df_solar['Generacion_Lag1'] = df_solar['Generacion_MW'].shift(1)
df_solar['Radiacion_Diff1'] = df_solar['Radiacion_Solar'].diff(1)
df_solar['Temp_Media_3h'] = df_solar['Temperatura_Ambiente'].rolling(window=3).mean()
df_solar['Nubes_Media_3h'] = df_solar['Nubosidad_Porcentaje'].rolling(window=3).mean()
df_solar.dropna(inplace=True)

X_s = df_solar.drop(columns=['Generacion_MW'])
y_s = df_solar['Generacion_MW']
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_s, y_s, test_size=0.2, shuffle=False)

mod_solar = XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42)
mod_solar.fit(X_tr_s, y_tr_s)
plot_feature_importance(mod_solar, X_s.columns, 'Importancia de Variables: Modelo Solar Fotovoltaico')

# =====================================================================
# 2. PROCESAMIENTO Y ENTRENAMIENTO: MODELO 2 (DEMANDA ELÉCTRICA)
# =====================================================================
print("\n--- PROCESANDO MODELO 2: DEMANDA ELÉCTRICA ---")
es_fin_semana = np.where(dias_semana >= 5, 1, 0)
demanda_base = 200 + np.sin((horas - 6) / 24 * 2 * np.pi) * 40 + np.sin((horas - 15) / 24 * 4 * np.pi) * 20
factor_calendario = np.where(es_fin_semana == 1, 0.75, 1.0)
efecto_termico = np.where(temperatura < 18, (18 - temperatura) ** 1.5 * 4, 0)
demanda_mw = (demanda_base * factor_calendario) + efecto_termico + np.random.normal(0, 5, len(fechas))

df_demanda = pd.DataFrame({
    'Temperatura_Ambiente': temperatura, 'Dia_Semana': dias_semana,
    'Es_Fin_Semana': es_fin_semana, 'Es_Festivo': es_festivo, 'Demanda_MW': demanda_mw
}, index=fechas)

df_demanda['Hora_Sin'] = np.sin(2 * np.pi * df_demanda.index.hour / 24.0)
df_demanda['Hora_Cos'] = np.cos(2 * np.pi * df_demanda.index.hour / 24.0)
df_demanda['Dia_Semana_Sin'] = np.sin(2 * np.pi * df_demanda['Dia_Semana'] / 7.0)
df_demanda['Dia_Semana_Cos'] = np.cos(2 * np.pi * df_demanda['Dia_Semana'] / 7.0)
df_demanda['Demanda_Lag1'] = df_demanda['Demanda_MW'].shift(1)
df_demanda['Demanda_Lag2'] = df_demanda['Demanda_MW'].shift(2)
df_demanda['Demanda_Lag24'] = df_demanda['Demanda_MW'].shift(24)
df_demanda['Temp_Lag1'] = df_demanda['Temperatura_Ambiente'].shift(1)
df_demanda['Temp_Media_4h'] = df_demanda['Temperatura_Ambiente'].rolling(window=4).mean()
df_demanda.dropna(inplace=True)

X_d = df_demanda.drop(columns=['Demanda_MW'])
y_d = df_demanda['Demanda_MW']
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.2, shuffle=False)

mod_demanda = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
mod_demanda.fit(X_tr_d, y_tr_d)
plot_feature_importance(mod_demanda, X_d.columns, 'Importancia de Variables: Modelo de Demanda')

# =====================================================================
# 3. PROCESAMIENTO Y ENTRENAMIENTO: MODELO 3 (PRECIOS DE MERCADO)
# =====================================================================
print("\n--- PROCESANDO MODELO 3: PRECIOS DE MERCADO ---")
margen_reserva = demanda_mw - generacion_solar
precio_gas = 35 + np.cumsum(np.random.normal(0, 0.4, len(fechas)))
precio_co2 = 70 + np.cumsum(np.random.normal(0, 0.2, len(fechas)))

precio_base_gas = (precio_gas * 2) + (precio_co2 * 0.4)
componente_escasez = np.where(margen_reserva > 180, ((margen_reserva - 180) ** 1.8) * 1.5, 0)
componente_solar = np.where((generacion_solar > 80) & (demanda_mw < 200), -40, 0)
precio_pool = np.clip(precio_base_gas + componente_escasez + componente_solar + np.random.normal(0, 4, len(fechas)), 0, None)

df_precios = pd.DataFrame({
    'Demanda_MW': demanda_mw, 'Generacion_Solar_MW': generacion_solar, 'Margen_Reserva_MW': margen_reserva,
    'Precio_Gas_Euros_MWh': precio_gas, 'Precio_CO2_Euros_Ton': precio_co2, 'Precio_Pool_Euros_MWh': precio_pool
}, index=fechas)

df_precios['Hora_Sin'] = np.sin(2 * np.pi * df_precios.index.hour / 24.0)
df_precios['Hora_Cos'] = np.cos(2 * np.pi * df_precios.index.hour / 24.0)
df_precios['Precio_Lag1'] = df_precios['Precio_Pool_Euros_MWh'].shift(1)
df_precios['Precio_Lag2'] = df_precios['Precio_Pool_Euros_MWh'].shift(2)
df_precios['Precio_Lag24'] = df_precios['Precio_Pool_Euros_MWh'].shift(24)
df_precios['Variacion_Margen_3h'] = df_precios['Margen_Reserva_MW'].diff(3)
df_precios.dropna(inplace=True)

X_p = df_precios.drop(columns=['Precio_Pool_Euros_MWh'])
y_p = df_precios['Precio_Pool_Euros_MWh']
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_p, y_p, test_size=0.2, shuffle=False)

mod_precios = XGBRegressor(n_estimators=250, learning_rate=0.04, max_depth=6, random_state=42)
mod_precios.fit(X_tr_p, y_tr_p)
plot_feature_importance(mod_precios, X_p.columns, 'Importancia de Variables: Modelo de Precios Pool')

# =====================================================================
# 4. SERIALIZACIÓN Y EXPORTACIÓN PERSISTENTE (.PKL)
# =====================================================================
print("\n--- INICIANDO EXPORTACIÓN PERSISTENTE A DRIVE ---")

# Diccionario con rutas y objetos para optimizar el bucle de guardado
modelos_produccion = {
    'modelo_solar_v1.pkl': mod_solar,
    'modelo_demanda_v1.pkl': mod_demanda,
    'modelo_precios_v1.pkl': mod_precios
}

for nombre_archivo, objeto_modelo in modelos_produccion.items():
    ruta_guardado = os.path.join(ruta_drive, nombre_archivo)

    # joblib con compress=3 optimiza el tamaño del archivo binario comprimiendo los árboles de XGBoost
    joblib.dump(objeto_modelo, ruta_guardado, compress=3)
    print(f"Éxito: {nombre_archivo} exportado a production-ready en: {ruta_guardado}")

print("\n=======================================================")
print("¡Pipeline de producción completado y guardado en Drive!")
print("=======================================================")

Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- PROCESANDO MODELO 1: GENERACIÓN SOLAR ---



--- PROCESANDO MODELO 2: DEMANDA ELÉCTRICA ---



--- PROCESANDO MODELO 3: PRECIOS DE MERCADO ---



--- INICIANDO EXPORTACIÓN PERSISTENTE A DRIVE ---
Éxito: modelo_solar_v1.pkl exportado a production-ready en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/modelo_solar_v1.pkl
Éxito: modelo_demanda_v1.pkl exportado a production-ready en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/modelo_demanda_v1.pkl
Éxito: modelo_precios_v1.pkl exportado a production-ready en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/modelo_precios_v1.pkl

¡Pipeline de producción completado y guardado en Drive!


In [18]:
import os

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'

if os.path.exists(ruta_drive):
    archivos = os.listdir(ruta_drive)
    print("=== ARCHIVOS REALES EN TU CARPETA ===")
    for archivo in archivos:
        print(f"-> {archivo}")
else:
    print("La ruta especificada no existe todavía.")


=== ARCHIVOS REALES EN TU CARPETA ===
-> Generación_Energia_Renovable.ipynb
-> datos_solares_avanzados.csv
-> datos_demanda_avanzados.csv
-> datos_precios_mercado.csv
-> modelo_solar_v1.pkl
-> modelo_demanda_v1.pkl
-> modelo_precios_v1.pkl


In [28]:
import os
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from google.colab import drive
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)

# Fijamos la semilla de NumPy para la generación estable de los datasets
np.random.seed(42)
fechas = pd.date_range(start="2026-06-01", end="2026-06-30", freq="h")
horas = fechas.hour
dias_semana = fechas.dayofweek

# FUNCIÓN AUXILIAR: RENDERIZADO DE GRÁFICAS (PLOTLY) CON % FIJO OUTSIDE
def mostrar_importancia_variables(modelo, columnas, titulo_grafica):
    importancias = modelo.feature_importances_
    indices_imp = np.argsort(importancias)
    valores_porcentaje = importancias[indices_imp] * 100

    df_imp = pd.DataFrame({
        'Variable': [columnas[i] for i in indices_imp],
        'Importancia': valores_porcentaje
    })

    etiquetas_fijas = [f"{v:.4f}%" for v in valores_porcentaje]

    fig = px.bar(
        df_imp, x='Importancia', y='Variable', orientation='h',
        title=titulo_grafica, color='Importancia',
        color_continuous_scale='Viridis', text=etiquetas_fijas
    )
    fig.update_traces(
        textposition='outside',
        hovertemplate='Variable: %{y}<br>Importancia: %{x:.4f}%<extra></extra>'
    )
    fig.update_layout(
        template='plotly_white', coloraxis_showscale=False,
        xaxis=dict(ticksuffix="%", range=[0, max(valores_porcentaje) * 1.15])
    )
    fig.show()

# =====================================================================
# 1. PIPELINE DEL MODELO 1: GENERACIÓN SOLAR FOTOVOLTAICA
# =====================================================================
print("\n☀️ [1/3] Procesando e Ingeniería de Variables: Modelo Solar...")
radiacion = np.where((horas > 5) & (horas < 20), np.sin((horas - 5) / 14 * np.pi) * 900 + np.random.normal(0, 40, len(fechas)), 0)
radiacion = np.clip(radiacion, 0, None)
temperatura = 22 + np.sin((horas - 8) / 24 * 2 * np.pi) * 12 + np.random.normal(0, 1.5, len(fechas))
viento = np.abs(np.random.normal(4, 2, len(fechas)))
nubosidad = np.clip(np.random.beta(2, 5, len(fechas)) * 100 + np.where(radiacion == 0, 20, 0), 0, 100)
humedad = np.clip(100 - (temperatura * 2) - (radiacion * 0.03) + np.random.normal(0, 5, len(fechas)), 10, 100)

eficiencia = 0.16 - (temperatura - 25) * 0.003
generacion_solar = (radiacion * (1 - nubosidad/100) * eficiencia * 0.06) + np.random.normal(0, 0.3, len(fechas))
generacion_solar = np.where(radiacion == 0, 0, np.clip(generacion_solar, 0, None))

df_solar = pd.DataFrame({
    'Radiacion_Solar': radiacion, 'Temperatura_Ambiente': temperatura, 'Velocidad_Viento': viento,
    'Nubosidad_Porcentaje': nubosidad, 'Humedad_Relativa': humedad, 'Generacion_MW': generacion_solar
}, index=fechas)

df_solar['Hora_Sin'] = np.sin(2 * np.pi * df_solar.index.hour / 24.0)
df_solar['Hora_Cos'] = np.cos(2 * np.pi * df_solar.index.hour / 24.0)
df_solar['Mes_Sin'] = np.sin(2 * np.pi * df_solar.index.month / 12.0)
df_solar['Mes_Cos'] = np.cos(2 * np.pi * df_solar.index.month / 12.0)
df_solar['Radiacion_Lag1'] = df_solar['Radiacion_Solar'].shift(1)
df_solar['Radiacion_Lag2'] = df_solar['Radiacion_Solar'].shift(2)
df_solar['Generacion_Lag1'] = df_solar['Generacion_MW'].shift(1)
df_solar['Radiacion_Diff1'] = df_solar['Radiacion_Solar'].diff(1)
df_solar['Temp_Media_3h'] = df_solar['Temperatura_Ambiente'].rolling(window=3).mean()
df_solar['Nubes_Media_3h'] = df_solar['Nubosidad_Porcentaje'].rolling(window=3).mean()
df_solar.dropna(inplace=True)

features_solar = ['Radiacion_Solar', 'Temperatura_Ambiente', 'Velocidad_Viento', 'Nubosidad_Porcentaje',
                  'Humedad_Relativa', 'Hora_Sin', 'Hora_Cos', 'Mes_Sin', 'Mes_Cos', 'Radiacion_Lag1',
                  'Radiacion_Lag2', 'Generacion_Lag1', 'Radiacion_Diff1', 'Temp_Media_3h', 'Nubes_Media_3h']

X_s = df_solar[features_solar]
y_s = df_solar['Generacion_MW']
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_s, y_s, test_size=0.2, shuffle=False)

mod_solar = XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42)
mod_solar.fit(X_tr_s, y_tr_s)
mostrar_importancia_variables(mod_solar, features_solar, 'Importancia de Variables: Modelo Solar Fotovoltaico')

# =====================================================================
# 2. PIPELINE DEL MODELO 2: DEMANDA ELÉCTRICA
# =====================================================================
print("\n📉 [2/3] Procesando e Ingeniería de Variables: Modelo Demanda...")
es_fin_semana = np.where(dias_semana >= 5, 1, 0)
demanda_base = 200 + np.sin((horas - 6) / 24 * 2 * np.pi) * 40 + np.sin((horas - 15) / 24 * 4 * np.pi) * 20
factor_calendario = np.where(es_fin_semana == 1, 0.75, 1.0)
efecto_termico = np.where(temperatura < 18, (18 - temperatura) ** 1.5 * 4, 0)
demanda_mw = (demanda_base * factor_calendario) + efecto_termico + np.random.normal(0, 5, len(fechas))

df_demanda = pd.DataFrame({
    'Temperatura_Ambiente': temperatura, 'Dia_Semana': dias_semana,
    'Es_Fin_Semana': es_fin_semana, 'Demanda_MW': demanda_mw
}, index=fechas)

df_demanda['Hora_Sin'] = np.sin(2 * np.pi * df_demanda.index.hour / 24.0)
df_demanda['Hora_Cos'] = np.cos(2 * np.pi * df_demanda.index.hour / 24.0)
df_demanda['Demanda_Lag1'] = df_demanda['Demanda_MW'].shift(1)
df_demanda['Demanda_Lag2'] = df_demanda['Demanda_MW'].shift(2)
df_demanda['Demanda_Lag24'] = df_demanda['Demanda_MW'].shift(24)
df_demanda['Temp_Media_4h'] = df_demanda['Temperatura_Ambiente'].rolling(window=4).mean()
df_demanda.dropna(inplace=True)

features_demanda = ['Temperatura_Ambiente', 'Dia_Semana', 'Es_Fin_Semana', 'Hora_Sin',
                    'Hora_Cos', 'Demanda_Lag1', 'Demanda_Lag2', 'Demanda_Lag24', 'Temp_Media_4h']

X_d = df_demanda[features_demanda]
y_d = df_demanda['Demanda_MW']
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.2, shuffle=False)

mod_demanda = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
mod_demanda.fit(X_tr_d, y_tr_d)
mostrar_importancia_variables(mod_demanda, features_demanda, 'Importancia de Variables: Modelo de Demanda')

# =====================================================================
# 3. PIPELINE DEL MODELO 3: PRECIOS DE MERCADO POOL
# =====================================================================
print("\n💶 [3/3] Procesando e Ingeniería de Variables: Modelo Precios Pool...")
margen_reserva = demanda_mw - generacion_solar
precio_gas = 35 + np.cumsum(np.random.normal(0, 0.4, len(fechas)))
precio_co2 = 70 + np.cumsum(np.random.normal(0, 0.2, len(fechas)))

precio_base_gas = (precio_gas * 2) + (precio_co2 * 0.4)
componente_escasez = np.where(margen_reserva > 180, ((margen_reserva - 180) ** 1.8) * 1.5, 0)
componente_solar = np.where((generacion_solar > 80) & (demanda_mw < 200), -40, 0)
precio_pool = np.clip(precio_base_gas + componente_escasez + componente_solar + np.random.normal(0, 4, len(fechas)), 0, None)

df_precios = pd.DataFrame({
    'Demanda_MW': demanda_mw, 'Generacion_Solar_MW': generacion_solar, 'Margen_Reserva_MW': margen_reserva,
    'Precio_Gas_Euros_MWh': precio_gas, 'Precio_CO2_Euros_Ton': precio_co2, 'Precio_Pool_Euros_MWh': precio_pool
}, index=fechas)

df_precios['Hora_Sin'] = np.sin(2 * np.pi * df_precios.index.hour / 24.0)
df_precios['Hora_Cos'] = np.cos(2 * np.pi * df_precios.index.hour / 24.0)
df_precios['Precio_Lag1'] = df_precios['Precio_Pool_Euros_MWh'].shift(1)
df_precios['Precio_Lag2'] = df_precios['Precio_Pool_Euros_MWh'].shift(2)
df_precios['Precio_Lag24'] = df_precios['Precio_Pool_Euros_MWh'].shift(24)
df_precios['Variacion_Margen_3h'] = df_precios['Margen_Reserva_MW'].diff(3)
df_precios.dropna(inplace=True)

features_precios = ['Demanda_MW', 'Generacion_Solar_MW', 'Margen_Reserva_MW', 'Precio_Gas_Euros_MWh',
                    'Precio_CO2_Euros_Ton', 'Hora_Sin', 'Hora_Cos', 'Precio_Lag1', 'Precio_Lag2',
                    'Precio_Lag24', 'Variacion_Margen_3h']

X_p = df_precios[features_precios]
y_p = df_precios['Precio_Pool_Euros_MWh']
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_p, y_p, test_size=0.2, shuffle=False)

mod_precios = XGBRegressor(n_estimators=250, learning_rate=0.04, max_depth=6, random_state=42)
mod_precios.fit(X_tr_p, y_tr_p)
mostrar_importancia_variables(mod_precios, features_precios, 'Importancia de Variables: Modelo de Precios')

# =====================================================================
# 4. SERIALIZACIÓN PERSISTENTE A GOOGLE DRIVE (.PKL)
# =====================================================================
print("\n💾 Guardando los modelos entrenados en tu Google Drive...")
modelos_dict = {
    'modelo_solar_v1.pkl': mod_solar,
    'modelo_demanda_v1.pkl': mod_demanda,
    'modelo_precios_v1.pkl': mod_precios
}

for archivo, modelo_obj in modelos_dict.items():
    ruta_exportacion = os.path.join(ruta_drive, archivo)
    joblib.dump(modelo_obj, ruta_exportacion, compress=3)
    print(f" -> Exportado: '{archivo}' guardado exitosamente en Drive.")

print("\n🎉 ¡ENTRENAMIENTO Y SERIALIZACIÓN FINALIZADOS CON ÉXITO! Pasa al bloque 2.")


🔌 Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

☀️ [1/3] Procesando e Ingeniería de Variables: Modelo Solar...



📉 [2/3] Procesando e Ingeniería de Variables: Modelo Demanda...



💶 [3/3] Procesando e Ingeniería de Variables: Modelo Precios Pool...



💾 Guardando los modelos entrenados en tu Google Drive...
 -> Exportado: 'modelo_solar_v1.pkl' guardado exitosamente en Drive.
 -> Exportado: 'modelo_demanda_v1.pkl' guardado exitosamente en Drive.
 -> Exportado: 'modelo_precios_v1.pkl' guardado exitosamente en Drive.

🎉 ¡ENTRENAMIENTO Y SERIALIZACIÓN FINALIZADOS CON ÉXITO! Pasa al bloque 2.


In [29]:
import os
import joblib
import numpy as np
import pandas as pd
from google.colab import drive

# =====================================================================
# 0. CONFIGURACIÓN Y CARGA DE MODELOS DESDE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive para cargar modelos...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'

# Carga rápida de binarios
mod_solar = joblib.load(os.path.join(ruta_drive, 'modelo_solar_v1.pkl'))
mod_demanda = joblib.load(os.path.join(ruta_drive, 'modelo_demanda_v1.pkl'))
mod_precios = joblib.load(os.path.join(ruta_drive, 'modelo_precios_v1.pkl'))
print("✅ ¡Los 3 modelos (.pkl) se cargaron en memoria correctamente!")

# Listas oficiales de características mapeadas en entrenamiento
features_solar = ['Radiacion_Solar', 'Temperatura_Ambiente', 'Velocidad_Viento', 'Nubosidad_Porcentaje',
                  'Humedad_Relativa', 'Hora_Sin', 'Hora_Cos', 'Mes_Sin', 'Mes_Cos', 'Radiacion_Lag1',
                  'Radiacion_Lag2', 'Generacion_Lag1', 'Radiacion_Diff1', 'Temp_Media_3h', 'Nubes_Media_3h']

features_demanda = ['Temperatura_Ambiente', 'Dia_Semana', 'Es_Fin_Semana', 'Hora_Sin',
                    'Hora_Cos', 'Demanda_Lag1', 'Demanda_Lag2', 'Demanda_Lag24', 'Temp_Media_4h']

features_precios = ['Demanda_MW', 'Generacion_Solar_MW', 'Margen_Reserva_MW', 'Precio_Gas_Euros_MWh',
                    'Precio_CO2_Euros_Ton', 'Hora_Sin', 'Hora_Cos', 'Precio_Lag1', 'Precio_Lag2',
                    'Precio_Lag24', 'Variacion_Margen_3h']

# =====================================================================
# 1. SIMULACIÓN DE DATOS DE ENTRADA EN PRODUCCIÓN
# =====================================================================
print("\n🚀 Recibiendo pronósticos operativos (Próximas 3 Horas)...")
fechas_futuras = pd.date_range(start="2026-07-01 12:00:00", periods=3, freq="h")

# DataFrame de Entrada (Simula la lectura de APIs meteorológicas y financieras reales)
datos_nuevos = pd.DataFrame({
    'Radiacion_Solar': [850.0, 910.0, 800.0],
    'Temperatura_Ambiente': [29.0, 31.5, 32.0],
    'Velocidad_Viento': [3.8, 4.0, 4.5],
    'Nubosidad_Porcentaje': [8.0, 12.0, 10.0],
    'Humedad_Relativa': [28.0, 24.0, 25.0],
    'Precio_Gas_Euros_MWh': [39.10, 39.10, 39.10],
    'Precio_CO2_Euros_Ton': [73.20, 73.20, 73.20],

    # Rezagos fijos del sistema recopilados en las últimas 24 horas
    'Radiacion_Lag1': [720.0, 850.0, 910.0],
    'Radiacion_Lag2': [510.0, 720.0, 850.0],
    'Generacion_Lag1': [22.1, 26.4, 29.0],
    'Demanda_Lag1': [215.0, 235.0, 250.0],
    'Demanda_Lag2': [200.0, 215.0, 235.0],
    'Demanda_Lag24': [218.0, 230.0, 244.0],
    'Temp_Lag1': [27.2, 29.0, 31.5],
    'Precio_Lag1': [86.50, 91.20, 95.00],
    'Precio_Lag2': [82.00, 86.50, 91.20],
    'Precio_Lag24': [89.00, 92.00, 94.10]
}, index=fechas_futuras)

# =====================================================================
# 2. INGENIERÍA DE VARIABLES EN TIEMPO REAL (ON-THE-FLY)
# =====================================================================
datos_nuevos['Hora_Sin'] = np.sin(2 * np.pi * datos_nuevos.index.hour / 24.0)
datos_nuevos['Hora_Cos'] = np.cos(2 * np.pi * datos_nuevos.index.hour / 24.0)
datos_nuevos['Mes_Sin'] = np.sin(2 * np.pi * datos_nuevos.index.month / 12.0)
datos_nuevos['Mes_Cos'] = np.cos(2 * np.pi * datos_nuevos.index.month / 12.0)
datos_nuevos['Dia_Semana'] = datos_nuevos.index.dayofweek
datos_nuevos['Es_Fin_Semana'] = np.where(datos_nuevos['Dia_Semana'] >= 5, 1, 0)

datos_nuevos['Radiacion_Diff1'] = datos_nuevos['Radiacion_Solar'] - datos_nuevos['Radiacion_Lag1']
datos_nuevos['Temp_Media_3h'] = (datos_nuevos['Temperatura_Ambiente'] + datos_nuevos['Temp_Lag1'] + datos_nuevos['Temperatura_Ambiente'].shift(1).fillna(27.2)) / 3
datos_nuevos['Nubes_Media_3h'] = datos_nuevos['Nubosidad_Porcentaje']
datos_nuevos['Temp_Media_4h'] = datos_nuevos['Temp_Media_3h']

# =====================================================================
# 3. PIPELINE SECUENCIAL Y CONSOLIDACIÓN DEL REPORTE
# =====================================================================
print("⚙️ Ejecutando cadena acoplada de inferencia...")

# Inferencia secuencial obligatoria
pred_solar_prod = mod_solar.predict(datos_nuevos[features_solar])
pred_demanda_prod = mod_demanda.predict(datos_nuevos[features_demanda])

# Inyección cruzada de variables calculadas para el modelo financiero
datos_nuevos['Demanda_MW'] = pred_demanda_prod
datos_nuevos['Generacion_Solar_MW'] = pred_solar_prod
datos_nuevos['Margen_Reserva_MW'] = pred_demanda_prod - pred_solar_prod
datos_nuevos['Variacion_Margen_3h'] = datos_nuevos['Margen_Reserva_MW'].diff(3).fillna(0.0)

# Inferencia del Modelo de Precios final
pred_precios_prod = mod_precios.predict(datos_nuevos[features_precios])

# Construcción final del dataframe de negocio
reporte_final = pd.DataFrame({
    '☀️ Generación Solar (MW)': np.round(pred_solar_prod, 2),
    '📉 Demanda Sistema (MW)': np.round(pred_demanda_prod, 2),
    '💶 Precio Pool Estimado (€/MWh)': np.round(pred_precios_prod, 2)
}, index=fechas_futuras.strftime('%H:%M h'))

print("\n📊 ==================== REPORTE GENERADO ====================")
print(reporte_final.to_string())
print("============================================================\n")
print("🎯 Pipeline finalizado sin errores. Datos listos en Drive.")


🔌 Conectando con Google Drive para cargar modelos...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ¡Los 3 modelos (.pkl) se cargaron en memoria correctamente!

🚀 Recibiendo pronósticos operativos (Próximas 3 Horas)...
⚙️ Ejecutando cadena acoplada de inferencia...

📊 ==================== REPORTE GENERADO ====================
         ☀️ Generación Solar (MW)  📉 Demanda Sistema (MW)  💶 Precio Pool Estimado (€/MWh)
12:00 h                      6.75              219.479996                      880.609985
13:00 h                      7.17              223.100006                     1034.650024
14:00 h                      5.28              225.520004                     1251.420044

🎯 Pipeline finalizado sin errores. Datos listos en Drive.


In [21]:
texto_markdown_anterior = """# Proyectos de Modelos ML para Petición de Trabajo\n\nEste repositorio contiene una serie de proyectos de Machine Learning desarrollados como parte de un portafolio para búsqueda de empleo. Cada proyecto aborda un problema diferente en el ámbito de la energía y utiliza modelos predictivos.\n\n## Estructura del Repositorio\n\n- `datos_solares_avanzados.csv`: Datos sintéticos de generación solar.\n- `datos_demanda_avanzados.csv`: Datos sintéticos de demanda eléctrica.\n- `datos_precios_mercado.csv`: Datos sintéticos de precios del mercado eléctrico.\n- `modelo_solar_v1.pkl`: Modelo entrenado para la predicción de generación solar.\n- `modelo_demanda_v1.pkl`: Modelo entrenado para la predicción de demanda eléctrica.\n- `modelo_precios_v1.pkl`: Modelo entrenado para la predicción de precios del mercado.\n\n## Modelos Implementados\n\n1.  **Predicción de Generación Solar Fotovoltaica**: Utiliza variables meteorológicas para predecir la producción de energía solar.\n2.  **Predicción de Demanda Eléctrica**: Estima la carga de la red eléctrica considerando factores horarios y estacionales.\n3.  **Predicción de Precios del Mercado Eléctrico (Pool)**: Pronostica los precios de la electricidad basándose en la oferta, demanda, y precios de commodities como el gas y el CO2.\n\n## Herramientas y Tecnologías\n\n- Python\n- Pandas (manipulación de datos)\n- NumPy (operaciones numéricas)\n- Scikit-learn (preprocesamiento, métricas)\n- XGBoost (modelado predictivo)\n- Matplotlib y Seaborn (visualización estática)\n- Plotly (visualización interactiva)\n- Google Colab (entorno de desarrollo)\n- Google Drive (almacenamiento de datos y modelos)\n\n## Uso\n\nLos notebooks de Colab (`.ipynb`) contienen el código fuente para la generación de datos, feature engineering, entrenamiento y evaluación de cada modelo. Los modelos pre-entrenados (`.pkl`) se guardan directamente en Google Drive para su posterior despliegue o uso en producción.\n\nPara ejecutar los notebooks, es necesario montarlos en Google Colab y asegurarse de tener acceso a los archivos CSV generados en la ruta especificada en cada notebook.\n\n---\n\n**Contacto**: [Tu Nombre/Contacto Profesional]\n\n"""
ruta_readme = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/README.md'
with open(ruta_readme, 'w', encoding='utf-8') as f:
    f.write(texto_markdown_anterior)

In [ ]:
# Carga de los pronósticos climáticos reales entregados por la empresa
datos_nuevos = pd.read_csv('/content/drive/MyDrive/Ruta/pronosticos_manana.csv', parse_dates=['Fecha_Hora'])
datos_nuevos.set_index('Fecha_Hora', inplace=True)


In [ ]:
import requests

# Ejemplo conceptual de cómo tu script llamaría a la API real de la empresa
respuesta = requests.get("https://operadorelectrico.com", headers={"Authorization": "Bearer TOKEN"})
datos_nuevos = pd.DataFrame(respuesta.json()['data'])
